# 📊 SmartStock – Sales Analytics & Trend Forecasting
### Automated Python / Pandas / Matplotlib Data Pipeline
**Objective**: Analyze daily/monthly sales volume, growth rates, category revenue contribution, and sales seasonality trends using real SmartStock transactional datasets.


## 1. Environment Setup & Data Ingestion
Import analytical libraries and load exported sales records.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Define dataset paths
DATA_DIR = os.path.join('..', 'datasets')
sales_path = os.path.join(DATA_DIR, 'sales.csv')
items_path = os.path.join(DATA_DIR, 'sale_items.csv')

df_sales = pd.read_csv(sales_path)
df_items = pd.read_csv(items_path)

# Convert timestamps
df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date'])
df_sales['date'] = df_sales['sale_date'].dt.date
df_sales['month'] = df_sales['sale_date'].dt.to_period('M')
df_sales['day_of_week'] = df_sales['sale_date'].dt.day_name()
df_sales['hour'] = df_sales['sale_date'].dt.hour

df_items['sale_date'] = pd.to_datetime(df_items['sale_date'])

print(f"✅ Loaded {len(df_sales):,} sales transactions and {len(df_items):,} item line records.")
df_sales.head(3)


## 2. Key Sales Summary Metrics
Compute foundational top-line metrics: total revenue, order count, and Average Order Value (AOV).


In [ ]:
total_revenue = df_sales['total_amount'].sum()
total_orders = len(df_sales)
aov = df_sales['total_amount'].mean()
total_items_sold = df_items['quantity'].sum()

print("=" * 50)
print(f"💰 Total Sales Revenue:     ${total_revenue:,.2f}")
print(f"📦 Total Orders Placed:     {total_orders:,}")
print(f"🏷️  Average Order Value:    ${aov:,.2f}")
print(f"🛒 Total Units Sold:        {total_items_sold:,}")
print("=" * 50)


## 3. Daily Sales & Trend Moving Averages
Examine daily transactional patterns with 7-day and 30-day moving averages to smooth fluctuations.


In [ ]:
daily_sales = df_sales.groupby('date').agg(
    daily_revenue=('total_amount', 'sum'),
    order_count=('sale_id', 'count')
).reset_index()
daily_sales['date'] = pd.to_datetime(daily_sales['date'])
daily_sales['ma_7d'] = daily_sales['daily_revenue'].rolling(window=7, min_periods=1).mean()
daily_sales['ma_30d'] = daily_sales['daily_revenue'].rolling(window=30, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(daily_sales['date'], daily_sales['daily_revenue'], label='Daily Revenue', color='#3b82f6', alpha=0.45, lw=1.2)
plt.plot(daily_sales['date'], daily_sales['ma_7d'], label='7-Day Moving Avg', color='#2563eb', lw=2.2)
plt.plot(daily_sales['date'], daily_sales['ma_30d'], label='30-Day Moving Avg', color='#d97706', lw=2.5, linestyle='--')
plt.title('SmartStock Daily Revenue & Moving Averages (Trend Analysis)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Date', fontweight='semibold')
plt.ylabel('Revenue ($)', fontweight='semibold')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


## 4. Monthly Sales & Sales Growth Rates (MoM)
Calculate Month-over-Month (MoM) percentage sales growth and order velocity.


In [ ]:
monthly_sales = df_sales.groupby('month').agg(
    revenue=('total_amount', 'sum'),
    orders=('sale_id', 'count'),
    aov=('total_amount', 'mean')
).reset_index()

monthly_sales['mom_growth_pct'] = monthly_sales['revenue'].pct_change() * 100
monthly_sales['revenue_formatted'] = monthly_sales['revenue'].apply(lambda x: f"${x:,.2f}")
monthly_sales['growth_formatted'] = monthly_sales['mom_growth_pct'].apply(lambda x: f"{x:+.2f}%" if pd.notnull(x) else "Baseline")

display(monthly_sales[['month', 'revenue_formatted', 'orders', 'growth_formatted']])

# Plot Monthly Revenue and Growth Rate
fig, ax1 = plt.subplots(figsize=(12, 6))

x_labels = [str(m) for m in monthly_sales['month']]
bars = ax1.bar(x_labels, monthly_sales['revenue'], color='#0ea5e9', width=0.55, label='Monthly Revenue ($)', alpha=0.85)
ax1.set_ylabel('Monthly Revenue ($)', color='#0284c7', fontweight='semibold')
ax1.yaxis.set_major_formatter('${x:,.0f}')

# Overlay Growth Rate on Secondary Axis
ax2 = ax1.twinx()
ax2.plot(x_labels, monthly_sales['mom_growth_pct'], color='#ef4444', marker='o', linewidth=2.5, label='MoM Growth (%)')
ax2.set_ylabel('MoM Growth Rate (%)', color='#ef4444', fontweight='semibold')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.7)

plt.title('Monthly Sales Revenue & Month-over-Month (MoM) Growth', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
plt.show()


## 5. Category Performance Analysis
Break down gross sales and total item volume across product categories.


In [ ]:
cat_perf = df_items.groupby('category_name').agg(
    total_sales=('subtotal', 'sum'),
    units_sold=('quantity', 'sum'),
    unique_items=('product_id', 'nunique')
).reset_index().sort_values(by='total_sales', ascending=False)

cat_perf['sales_share_pct'] = (cat_perf['total_sales'] / cat_perf['total_sales'].sum()) * 100

plt.figure(figsize=(12, 6))
bars = plt.barh(cat_perf['category_name'][::-1], cat_perf['total_sales'][::-1], color='#6366f1', alpha=0.85)
plt.title('Revenue Contribution by Product Category', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Total Revenue ($)', fontweight='semibold')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')

# Add value tags
for bar in bars:
    w = bar.get_width()
    plt.text(w + 500, bar.get_y() + bar.get_height()/2, f"${w:,.0f}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

display(cat_perf[['category_name', 'total_sales', 'units_sold', 'sales_share_pct']])


## 6. Sales Trends: Day-of-Week & Hourly Heat Patterns
Determine peak store shopping hours and highest performing days of the week.


In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = df_sales.groupby('day_of_week')['total_amount'].agg(['sum', 'mean', 'count']).reindex(dow_order)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Day of week revenue
ax1.bar(dow_sales.index, dow_sales['sum'], color='#10b981', alpha=0.85)
ax1.set_title('Revenue by Day of Week', fontweight='bold')
ax1.set_ylabel('Total Revenue ($)')
ax1.yaxis.set_major_formatter('${x:,.0f}')
ax1.tick_params(axis='x', rotation=30)

# Hourly sales distribution
hourly_sales = df_sales.groupby('hour')['total_amount'].sum()
ax2.plot(hourly_sales.index, hourly_sales.values, marker='s', color='#8b5cf6', lw=2)
ax2.set_title('Hourly Sales Distribution (Store Trading Hours)', fontweight='bold')
ax2.set_xlabel('Hour of Day (24h format)')
ax2.set_ylabel('Total Revenue ($)')
ax2.yaxis.set_major_formatter('${x:,.0f}')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()
